In [ ]:
!git clone https://github.com/CryAndRRich/codapath.git

In [ ]:
%cd /kaggle/working/codapath
CODAPATH = "/kaggle/working/codapath"

In [ ]:
!pip install -r requirements.txt
!pip install -U huggingface_hub hf-transfer

In [ ]:
import os
from huggingface_hub import snapshot_download
from huggingface_hub import login

login("YOUR_HUGGINGFACE_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("Đang tải vinid/plip...")
snapshot_download(repo_id="vinid/plip")

print("Đang tải PubMedBERT...")
snapshot_download(repo_id="microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext")

print("Đang tải BiomedCLIP...")
snapshot_download(repo_id="microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")

In [5]:
import sys
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if CODAPATH not in sys.path:
    sys.path.append(CODAPATH)

In [6]:
import yaml
import torch

In [7]:
from run import main

In [8]:
PATHMNIST_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/pathmnist_224.npz"
HISTOSET_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/HistoSet-5x14/HistoSet-5x14"
SKINTISSUE_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/SkinTissue/SkinTissue/tiles"

DATA_DICT = {
    "pathmnist": PATHMNIST_PATH,
    "histoset": HISTOSET_PATH,
    "skintissue": SKINTISSUE_PATH
}

In [9]:
CONFIG_PATH = "config/config.yaml"

# pathmnist, histoset, skintissue
DATASET = "histoset"

# random, coreset, codapath
# entropy, margin, badge, typiclust, activeft
SAMPLER_NAME = "coreset"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

hyper = config.get("hyperparameters", {})
dataset_info = config["datasets"][DATASET]

In [ ]:
main(
    data_path=DATA_DICT[DATASET],
    sampler_name=SAMPLER_NAME,
    num_classes=dataset_info["num_classes"],
    cumulative_budget=config["cumulative_budget"],
    data_descriptions=dataset_info["descriptions"],
    prompt_templates=config["prompt_templates"],
    rank_lora=hyper["rank_lora"],
    num_epochs=hyper["num_epochs"],
    learn_rate=hyper["learning_rate"],
    alpha=hyper["alpha"],
    device=torch.device(config["device"]),
    random_seed=config["random_seed"],
    save_dir=f"checkpoints/{DATASET}",
    verbose=True
)